# 05 — Report pack: every figure and table for the report, in one folder

This notebook **does not compute anything new**. It takes the charts in `figures/` and the tables in `results/`
(created by notebooks 02, 03 and 04) and puts everything the report needs into the folder **`report_pack/`**:

| What | Where |
|---|---|
| The figures, **numbered in the order of the report** (`F01`, `F02`, … for the main text; `FA01`, … for the appendix) | `report_pack/figures/` |
| Every table, **rounded and formatted** (%, 2 decimals, p-values), one sheet per table, with a title and a note | `report_pack/tables/report_tables.xlsx` |
| The same tables as CSV (already formatted as text, ready to paste) and as Markdown | `report_pack/tables/*.csv`, `report_pack/tables/all_tables.md` |
| Every table as a **PNG image** (title, table and note, in the same colours as the charts), ready to insert in Word | `report_pack/tables/png/` |
| An index: for each figure and table, its caption, the section of the report where it goes and the notebook that made it | `report_pack/INDEX.md` |

The written guide to the results (key findings with the numbers, methods, limitations, recommendation and references) is in
`report_pack/REPORT_NOTES.md`; it is written by hand and is not created by this notebook.

The colours of all the charts and tables are set in one place, `notebooks/appm_style.py` (change a colour there and run the notebooks again).

**Run this notebook after notebooks 02, 03 and 04** (it only reads their outputs). It takes a few seconds.

In [1]:
import os
import shutil
import pandas as pd
from openpyxl.styles import Font, Alignment
from appm_style import apply_style, save_table_png     # the project's colour scheme (notebooks/appm_style.py)

apply_style()                                          # same fonts and colours as the charts of notebooks 02-04

RESULTS = '../results'
FIGURES = '../figures'
PACK = '../report_pack'

# figures/, tables/ and tables/png/ are recreated from scratch every time (so that no old file is left behind);
# REPORT_NOTES.md and README.md, written by hand, are in the main folder and are never touched
for folder in [f'{PACK}/figures', f'{PACK}/tables', f'{PACK}/tables/png']:
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)


def load(name):
    # Reads one table saved by notebooks 03/04 (they use 'utf-8-sig' so that Excel shows λ and σ correctly)
    return pd.read_csv(f'{RESULTS}/{name}.csv', encoding='utf-8-sig')

## 1. Figures: copy them with a number in the order of the report

The report follows the structure suggested in the brief (executive summary; data; methods and assumptions; results;
robustness checks and limitations; conclusions; references and reproducibility appendix) and the numbering of the group's draft
(1. universe, 2.1 stylised facts, 3. strategies). `Main` figures (F..) are meant for the main text, `Appendix` figures (FA..) for the appendix —
the group can move them as it prefers.

In [2]:
# One row per figure: (number, file in figures/, report section, caption, notebook and section that made it, Main/Appendix)
figure_list = [
    # --- 1. Data and investment universe
    ('F01', '04_universe_correlations.png', '1. Investment universe and data',
     'Correlation of the daily returns of the 23 selected stocks, 2010-2026 (Yahoo Finance adjusted close prices).',
     'notebook 04, section 2', 'Main'),
    ('F02', '04_risk_free_rate.png', '1. Data / 3. Assumptions',
     'Risk-free rate: 3-year US Treasury constant-maturity yield (FRED series DGS3), annual %, used as r_f (daily rate = annual / 252).',
     'notebook 04, section 2', 'Main'),
    # --- 2.1 Stylised facts (WMT)
    ('F03', 'day_log_returns.png', '2.1.1 Serial correlation / 2.1.4 Volatility clustering',
     'Daily log returns of Walmart (WMT), 2010-2026 (the same series is the top panel of F07: use one of the two).',
     'notebook 02', 'Main'),
    ('F04', '03_acf_returns_wmt.png', '2.1.1 Little or no serial correlation',
     'Autocorrelation function of WMT log returns at daily, weekly and monthly frequency, with 95% bands (±1.96/√N).',
     'notebook 03, section 3', 'Main'),
    ('F05', 'wmt_distribution_with_normal.png', '2.1.2 Non-normality and fat tails',
     'Histogram of WMT daily log returns with the fitted normal density (same mean and standard deviation).',
     'notebook 02', 'Main'),
    ('F06', '03_qqplot_normal_wmt_3freq.png', '2.1.2 Non-normality and fat tails / 2.1.3 Asymmetry',
     'Normal Q-Q plots of WMT log returns at daily, weekly and monthly frequency.',
     'notebook 03, section 4', 'Main'),
    ('F07', '03_rolling_volatility_wmt.png', '2.1.4 Volatility clustering',
     'WMT daily log returns (top) and their 21-day rolling volatility, annualised (bottom): calm and turbulent periods alternate.',
     'notebook 03, section 6', 'Main'),
    ('F08', '03_acf_abs_squared_wmt.png', '2.1.4 Volatility clustering',
     'Autocorrelation of absolute and squared WMT log returns at daily, weekly and monthly frequency.',
     'notebook 03, section 6', 'Main'),
    ('F09', '03_news_impact_curve.png', '2.1.5 Leverage effect',
     'News impact curve of the daily GJR-GARCH(1,1) model with Student-t innovations, WMT and SPY: next-day variance after a negative vs a positive shock of the same size.',
     'notebook 03, section 7', 'Main'),
    ('F10', '03_leverage_correlation.png', '2.1.5 Leverage effect',
     'Correlation between the daily return r(t) and the absolute return k days later |r(t+k)|, WMT and SPY (negative = leverage effect).',
     'notebook 03, section 7', 'Main'),
    ('F11', '03_conditional_volatility_wmt.png', '2.1.5 Leverage effect / 2.1.4 Volatility clustering',
     'Conditional volatility of WMT from the daily GJR-GARCH(1,1) model with Student-t innovations (annualised), with the rolling 21-day volatility for comparison.',
     'notebook 03, section 7', 'Main'),
    ('F12', '03_qqplot_std_residuals_wmt.png', '2.1.6 Conditional non-normality',
     'Q-Q plots of the standardised residuals of the GJR-GARCH model against the normal and the Student-t distribution, WMT, three frequencies.',
     'notebook 03, section 8', 'Main'),
    # --- 3. Strategies and backtest design
    ('F13', '04_weights_illustration.png', '3. Strategies: objectives and constraints',
     'Weights of each strategy estimated with the last 12 months of data on the 23 stocks (illustration, not a backtest result).',
     'notebook 04, section 4', 'Main'),
    ('F14', '04_risk_contributions_iv_vs_erc.png', '3. Strategies: inverse volatility vs equal risk contributions',
     'Risk contributions (% of portfolio risk) of inverse volatility and risk parity (ERC), 23 stocks, last 12 months: with correlated assets inverse volatility does not equalise risk contributions (line = 1/N = 4.35%; weights in table TA05).',
     'notebook 04, section 4', 'Main'),
    ('F15', '04_efficient_frontier.png', '3. Strategies: Markowitz vs GMV vs Max Sharpe',
     'In-sample efficient frontier (last 12 months, long-only) with the position of each strategy; Markowitz with λ = 3 is distinct from GMV and Max Sharpe.',
     'notebook 04, section 4', 'Main'),
    ('F16', '04_experiment_design.png', '3. Backtest design',
     'The 100 sampled 3-year periods (grey = first two years, used for estimation with a trailing 1-year window; blue = out-of-sample year) and how often each stock was drawn (seed 2026).',
     'notebook 04, section 5', 'Main'),
    # --- 3. Results
    ('F17', '04_boxplots_metrics.png', '3. Results: distribution across the 100 experiments',
     'Distribution across the 100 experiments of the Sharpe ratio, return, volatility, maximum drawdown, Sortino and Sterling ratios of the 10 portfolios (net of 10 bps).',
     'notebook 04, section 9.2', 'Main'),
    ('F18', '04_risk_return_and_beats_ew.png', '3. Results: risk-return and comparison with EW',
     'Left: median risk and return of each portfolio. Right: % of experiments in which each portfolio beats the equally weighted benchmark (EW).',
     'notebook 04, section 9.2', 'Main'),
    ('F19', '04_beats_group_pick_and_spy.png', '3. Results: comparison with the Group pick and SPY',
     '% of experiments in which each portfolio beats the Group pick (left) and SPY (right) on Sharpe, return, drawdown and Sterling.',
     'notebook 04, section 9.2', 'Main'),
    ('F20', '04_pairwise_sharpe.png', '3. Results: head-to-head comparison of all portfolios',
     'Head-to-head comparison: % of experiments in which the row portfolio has a higher net Sharpe ratio than the column portfolio.',
     'notebook 04, section 9.5', 'Main'),
    ('F21', '04_turnover_costs_concentration.png', '3. Results: turnover, costs and concentration',
     'Median annual turnover, cost drag (bps per year at 10 bps) and effective number of stocks of each portfolio.',
     'notebook 04, section 9.3', 'Main'),
    ('F22', '04_capm_beta_alpha.png', '3. Results: market risk (CAPM)',
     "CAPM beta and annualised Jensen's alpha of each portfolio against SPY, estimated in each out-of-sample year (100 experiments, net of costs).",
     'notebook 04, section 9.7', 'Main'),
    ('F23', '04_wealth_example.png', '3. Results: an example during a crisis',
     'Growth of 1 USD in the out-of-sample year of experiment 62 (11 Nov 2019 - 9 Nov 2020), which includes the March 2020 crash (net of costs).',
     'notebook 04, section 9.6', 'Main'),
    # --- Robustness
    ('F24', '04_cost_sensitivity.png', 'Robustness: transaction costs',
     'Median Sharpe ratio of each portfolio with 0, 10, 25 and 50 bps of transaction costs.',
     'notebook 04, section 10.1', 'Main'),
    ('F25', '04_sharpe_diff_vs_ew_over_time.png', 'Robustness: dependence on the period',
     'Sharpe ratio of each optimised strategy and of the Group pick minus that of EW in the same experiment, by start of the out-of-sample year.',
     'notebook 04, section 10.4', 'Main'),
    ('F26', '04_universe23_boxplot_sharpe.png', 'Robustness: the strategies on the group\'s 23-stock universe',
     'Net Sharpe ratio of the strategies applied to all 23 stocks (the same stocks as the Group pick), 100 experiments.',
     'notebook 04, section 11', 'Main'),
    ('F27', '04_fullsample_wealth.png', 'Robustness / limitations: full sample (hindsight bias)',
     'Growth of 1 USD, 2010-2026: Group pick vs SPY vs ^GSPC (log scale). Biased by hindsight: the 23 stocks were chosen in 2026.',
     'notebook 04, section 12', 'Main'),
    ('F28', '04_calendar_year_returns.png', 'Robustness / limitations: full sample (hindsight bias)',
     'Calendar-year returns of the Group pick, SPY and ^GSPC (2026 = year to date, until 18 September).',
     'notebook 04, section 12', 'Main'),
    # --- Appendix
    ('FA01', 'normalized_price_history.png', 'Appendix: data',
     'Normalised adjusted price of Walmart (initial value = 1), 2010-2026.', 'notebook 02', 'Appendix'),
    ('FA02', 'wmt_moving_averages.png', 'Appendix: data',
     'Walmart adjusted price with 30-day and 200-day moving averages.', 'notebook 02', 'Appendix'),
    ('FA03', 'week_log_returns.png', 'Appendix: stylised facts',
     'Weekly log returns of Walmart.', 'notebook 02', 'Appendix'),
    ('FA04', 'month_log_returns.png', 'Appendix: stylised facts',
     'Monthly log returns of Walmart.', 'notebook 02', 'Appendix'),
    ('FA05', 'wmt_log_returns_distribution.png', 'Appendix: stylised facts',
     'Distribution of WMT log returns at daily, weekly and monthly frequency.', 'notebook 02', 'Appendix'),
    ('FA06', 'wmt_log_returns_vs_normal.png', 'Appendix: stylised facts',
     'WMT daily log returns compared with simulated normal returns (Gaussian white noise) of the same mean and volatility.',
     'notebook 02', 'Appendix'),
    ('FA07', '04_pairwise_win_rates.png', 'Appendix: head-to-head comparison',
     'Head-to-head win rates of the 10 portfolios on Sharpe, return, maximum drawdown, volatility and Sterling, and the median paired Sharpe difference.',
     'notebook 04, section 9.5', 'Appendix'),
    ('FA08', '04_universe23_pairwise_sharpe.png', 'Appendix: the 23-stock universe',
     'Head-to-head Sharpe comparison of the strategies applied to the 23-stock universe.', 'notebook 04, section 11', 'Appendix'),
    ('FA09', '04_fullsample_drawdowns.png', 'Appendix: full sample',
     'Drawdowns of the Group pick, SPY and ^GSPC, 2010-2026.', 'notebook 04, section 12', 'Appendix'),
    ('FA10', '04_avg_weight_by_stock.png', 'Appendix: weights',
     'Average weight of each stock when it is drawn, by strategy (equal weight would be 8.3%).', 'notebook 04, section 10.5', 'Appendix'),
    ('FA11', '04_weights_illustration_heatmap.png', 'Appendix: strategies',
     'Weights of each strategy with the last 12 months of data on the 23 stocks, as a heatmap (same data as F13; darker = larger weight).',
     'notebook 04, section 4', 'Appendix'),
    ('FA12', '04_avg_weight_by_stock_bars.png', 'Appendix: weights',
     'Average weight of each stock when it is drawn, one panel per strategy (same data as FA10; line = equal weight 8.3%).',
     'notebook 04, section 10.5', 'Appendix'),
]

# Copy each figure with its number in front of the name, e.g. F17_04_boxplots_metrics.png
missing = []
for number, file_name, section, caption, made_by, place in figure_list:
    source = f'{FIGURES}/{file_name}'
    if os.path.exists(source):
        shutil.copyfile(source, f'{PACK}/figures/{number}_{file_name}')
    else:
        missing.append(file_name)

print('Figures copied:', len(figure_list) - len(missing))
print('Missing figures (run notebooks 02-04 first):', missing)

# Check that no figure of figures/ was forgotten in the list
listed = [row[1] for row in figure_list]
not_listed = [name for name in sorted(os.listdir(FIGURES)) if name.endswith('.png') and name not in listed]
print('Figures in figures/ that are not in the pack:', not_listed)

Figures copied: 40
Missing figures (run notebooks 02-04 first): []
Figures in figures/ that are not in the pack: []


## 2. Tables: rounded and formatted for the report

Each table is stored in a small dictionary with its number, title, the section of the report where it goes, a note and
**how each column should be shown**:

| Format | Meaning | Example |
|---|---|---|
| `percent` | the value is a fraction, shown as a % with 1 decimal | 0.356 → 35.6% |
| `percent2` | the same with 2 decimals (for small daily numbers) | 0.0005 → 0.05% |
| `whole` | a number shown without decimals (counts, or columns that are already "% of experiments") | 62.0 → 62 |
| `pvalue` | a p-value with 3 decimals ("<0.001" when it is smaller) | 0.0000 → <0.001 |
| `dec1`, `dec3`, `dec4` | 1, 3 or 4 decimals | |
| (not listed) | 2 decimals for numbers, text as it is | 1.7037 → 1.70 |

In [3]:
report_tables = []


def add_table(number, name, title, section, table, note='', percent=(), percent2=(), whole=(), pvalue=(), dec1=(), dec3=(), dec4=()):
    # Stores one table and the format of each column. 'table' must have normal columns (no index).
    formats = {}
    for kind, columns in [('percent', percent), ('percent2', percent2), ('whole', whole), ('pvalue', pvalue),
                          ('dec1', dec1), ('dec3', dec3), ('dec4', dec4)]:
        for column in columns:
            if column not in table.columns:
                raise KeyError(f'{number}: column {column!r} does not exist')
            formats[column] = kind
    report_tables.append({'number': number, 'name': name, 'title': title, 'section': section,
                          'table': table, 'note': note, 'formats': formats})


def first_column_as_name(table, name):
    # The tables saved with an index have an empty first column name: we give it a proper name
    return table.rename(columns={table.columns[0]: name})


PORTFOLIO = 'Portfolio'

### 2.1 Data and the stylised facts of Walmart (notebooks 03 and 04)

In [4]:
universe = load('04_universe_summary')
add_table('T01', 'universe', 'Investment universe: 23 S&P 500 stocks and SPY, 2010-2026',
          '1. Investment universe and data', universe,
          note='Daily adjusted close prices from Yahoo Finance, 4 Jan 2010 - 18 Sep 2026. TSLA starts on 30 Jun 2010 (IPO). '
               'CAGR, volatility and maximum drawdown are computed on simple returns over the full sample.',
          percent=['Ann. return (CAGR)', 'Ann. volatility', 'Max drawdown'])

descriptive = load('03_descriptive_stats')
add_table('T02', 'descriptive_wmt_spy', 'Descriptive statistics of WMT and SPY log returns at three frequencies',
          '2.1 Stylised facts', descriptive,
          note='Log returns from adjusted close prices. Weekly and monthly returns use period-end prices. Excess kurtosis = kurtosis - 3 (0 for a normal distribution).',
          percent2=['Mean', 'Std. dev.', 'Min', 'Max'], percent=['Ann. mean', 'Ann. volatility'], whole=['N'], dec3=['Skewness'])

facts_wmt = first_column_as_name(load('03_facts_summary_wmt'), 'Fact')
add_table('T03', 'facts_summary_wmt', 'Summary: is each stylised fact supported for WMT?',
          '2.1 Stylised facts (summary)', facts_wmt,
          note='Based on the tests of tables T04-T12 (5% significance level).')

facts_evidence = load('03_facts_summary').drop(columns=[load('03_facts_summary').columns[0]])
add_table('T04', 'facts_evidence', 'Evidence for each stylised fact: statistic, p-value and conclusion (WMT, with SPY for comparison)',
          '2.1 Stylised facts (summary)', facts_evidence,
          note='p-value column: 1) Ljung-Box on returns at lag 10 (daily), 8 (weekly), 6 (monthly); 2) Jarque-Bera; 3) bootstrap test of skewness; '
               '4) Ljung-Box on r^2 (the statistic column shows the Ljung-Box p-value on |r|; Partial = only one of the two is significant); '
               '5) gamma of the GJR-GARCH (Student-t); 6) Jarque-Bera of the standardised residuals.',
          pvalue=['p-value'])

acf = load('03_acf_lags_1_5')
add_table('T05', 'acf_returns', 'Autocorrelation of log returns, lags 1 to 5',
          '2.1.1 Little or no serial correlation', acf,
          note='Values outside ±1.96/√N are significant at 5%.',
          dec3=['lag 1', 'lag 2', 'lag 3', 'lag 4', 'lag 5', '95% band (±1.96/√N)'])

ljung_box = load('03_ljung_box_returns').drop(columns=['Unnamed: 0'])
add_table('T06', 'ljung_box_returns', 'Ljung-Box test on log returns (H0: no autocorrelation up to the lag)',
          '2.1.1 Little or no serial correlation', ljung_box,
          note='A rejection with very small autocorrelations (table T05) means statistically significant but economically negligible dependence.',
          whole=['Lag'], pvalue=['p-value'])

normality = load('03_normality')
add_table('T07', 'normality', 'Normality and fat tails: excess kurtosis, Jarque-Bera and frequency of 3-sigma days',
          '2.1.2 Non-normal distributions and fat tails', normality,
          note='"% |z|>3" = share of observations more than 3 standard deviations from the mean (0.27% under a normal distribution).',
          pvalue=['Kurtosis test p-value', 'JB p-value'], dec1=['Jarque-Bera'])

skewness = load('03_skewness')
add_table('T08', 'skewness', 'Asymmetry: skewness with a bootstrap test (robust to fat tails)',
          '2.1.3 Asymmetry and negative skewness', skewness,
          note='The classical skewtest assumes normal tails and over-rejects with fat tails; the bootstrap (resampling the data) does not. '
               'Bootstrap p = share of bootstrap samples with skewness >= 0.',
          dec3=['Skewness', 'Bootstrap 95% CI low', 'Bootstrap 95% CI high'], pvalue=['Skewtest p (assumes normal tails)', 'Bootstrap p (skew >= 0)'],
          whole=['N below -3σ', 'N above +3σ'], percent2=['Quantile 1%', 'Quantile 99%'])

vol_clustering = load('03_vol_clustering')
add_table('T09', 'vol_clustering', 'Volatility clustering: Ljung-Box on |r| and r^2 and ARCH-LM test',
          '2.1.4 Volatility clustering', vol_clustering,
          note='H0 = no autocorrelation in absolute / squared returns (no ARCH effects). Small p-values support volatility clustering.',
          whole=['Lag'], pvalue=['LB |r| p-value', 'LB r² p-value', 'ARCH-LM p-value'])

garch = load('03_gjr_garch')
add_table('T10', 'gjr_garch', 'GJR-GARCH(1,1) estimates with normal and Student-t innovations',
          '2.1.5 Leverage effect', garch,
          note='Returns in % (x100) for numerical stability. gamma > 0 and significant = negative shocks raise volatility more (leverage effect). '
               'The weekly-normal and monthly models are at the boundary of the parameter space (unreliable).',
          dec4=['omega', 'alpha', 'gamma', 'beta', 'persistence (α+γ/2+β)'], pvalue=['gamma p-value'], dec1=['Log-lik.', 'AIC', 'BIC'])

leverage = load('03_leverage_correlation')
add_table('T11', 'leverage_correlation', 'Correlation between r(t) and |r(t+k)| (daily)',
          '2.1.5 Leverage effect', leverage,
          note='Negative values = falls are followed by larger absolute moves (leverage effect).',
          whole=['k (days ahead)'], dec3=['WMT', 'SPY'])

std_residuals = load('03_std_residuals')
add_table('T12', 'std_residuals', 'Standardised residuals of the GJR-GARCH: skewness, excess kurtosis and tests',
          '2.1.6 Conditional non-normality', std_residuals,
          note='z = residual / conditional volatility. Excess kurtosis > 0 and a small Jarque-Bera p-value = conditional non-normality. '
               'LB z and LB z^2 check that the model removed the autocorrelation and the volatility clustering.',
          pvalue=['z JB p-value', 'LB z p-value', 'LB z² p-value'])

### 2.2 Strategies and backtest design (written here, from the conventions of notebook 04)

In [5]:
strategy_definitions = pd.DataFrame([
    ['a) Equally weighted (EW)', 'w_i = 1/N (N = 12 stocks drawn in each experiment)', 'nothing', 'Main benchmark of the brief'],
    ['b) Markowitz mean-variance', 'max w\'μ - (λ/2) w\'Σw', 'μ and Σ (trailing 1 year)', 'Risk aversion λ = 3 (sensitivity: λ = 1 and 5)'],
    ['c) Global minimum variance (GMV)', 'min w\'Σw', 'Σ', 'Ignores expected returns'],
    ['d) Max Sharpe', 'max (w\'μ - r_f) / √(w\'Σw)', 'μ, Σ and r_f (DGS3 of the last trading day before the rebalancing)', 'Tangency portfolio'],
    ['e) Inverse volatility', 'w_i ∝ 1/σ_i', 'σ_i only (ignores correlations)', 'No optimiser'],
    ['e) Risk parity (ERC)', 'equal risk contributions: w_i (Σw)_i / w\'Σw = 1/N', 'Σ (with correlations)', 'Weights > 0'],
    ['f) Most diversified', 'max DR = w\'σ / √(w\'Σw)', 'Σ', 'Highest diversification ratio'],
    ['f) Max decorrelation', 'min w\'Cw (C = correlation matrix)', 'correlations only', 'Minimum variance as if all volatilities were equal'],
    ['Group pick (23 stocks)', 'w_i = 1/N over all the group\'s stocks with data (23; 22 before TSLA)', 'nothing', 'The group\'s own selection; same rebalancing and costs'],
    ['SPY (buy & hold)', 'bought at the start of the evaluation year and held', 'nothing', 'Market benchmark (investable S&P 500 ETF); no costs'],
], columns=['Strategy', 'Objective / allocation rule', 'What is estimated', 'Parameters and notes'])
add_table('T13', 'strategy_definitions', 'The portfolios compared: objective, inputs and parameters',
          '3. Strategies: objectives and constraints', strategy_definitions,
          note='Common constraints for every strategy: long-only (w_i >= 0), fully invested (Σw_i = 1), no leverage. '
               'Optimisers: scipy SLSQP (ftol 1e-10, retry with 1e-8; equal weights if both fail - 0 failures in the main run).')

backtest_design = pd.DataFrame([
    ['Experiments', '100 contiguous three-year datasets (756 trading days) sampled at random, random seed 2026 (parameters fixed before the results were seen)'],
    ['Assets per experiment', '12 stocks drawn at random from the 23 (only stocks with data in the whole window); the Group pick holds all of them'],
    ['Split', 'Days 1-504: estimation period; days 505-756: out-of-sample evaluation (252 days). With the trailing 1-year window, '
              'the warm-up weights (held from day 379) use days 127-378 and the evaluation weights use days 253-504 and 379-630; only days 505-756 enter the metrics'],
    ['Estimation window', 'Trailing 1 year (252 days), only data available before each rebalancing'],
    ['Rebalancing', 'Semi-annual (every 126 trading days), first rebalancing one semester before the evaluation year; weights drift between rebalancings'],
    ['Trading convention', 'Weights set with data up to the close of day t-1 and traded at that close (market-on-close), so they apply from day t; '
                           'a one-day execution lag would be stricter'],
    ['Returns', 'Simple returns for portfolios and wealth; log returns only for the stylised facts'],
    ['Transaction costs', '10 bps per unit of two-way turnover Σ|Δw| (switching the whole portfolio costs 20 bps; robustness: 0, 25, 50 bps); '
                          'the initial purchase happens at the warm-up rebalancing, before the evaluation year; SPY buy & hold has no costs'],
    ['Risk-free rate', '3-year US Treasury yield (FRED DGS3), daily rate = annual / 252'],
    ['Metrics', 'Annualised return (geometric), volatility, Sharpe, Sortino, maximum drawdown, Calmar, Sterling = return / (|MDD| + 10%), turnover, effective N, diversification ratio, CAPM beta and alpha'],
    ['Summary across experiments', 'Medians, dispersion (P25/P75), % of experiments that beat EW / the Group pick / SPY, ranks and head-to-head win rates (no significance tests: the windows overlap)'],
    ['Robustness', 'Costs 0-50 bps; quarterly rebalancing; λ = 1, 3, 5; sub-periods and market regimes; new draw of stocks and all eligible stocks; strategies on the 23-stock universe'],
], columns=['Item', 'Convention'])
add_table('T14', 'backtest_design', 'Backtest design and conventions (the same for every strategy)',
          '3. Backtest design and assumptions', backtest_design)

### 2.3 Results of the backtest (notebook 04)

In [6]:
medians = load('04_median_metrics')
medians['Cost drag (bps)'] = medians['Cost drag'] * 10000          # from a fraction to basis points per year
medians = medians.drop(columns=['Cost drag']).rename(columns={'Strategy': PORTFOLIO})
add_table('T15', 'median_metrics', 'Median performance of the 10 portfolios across the 100 experiments (net of 10 bps)',
          '3. Results', medians,
          note='Medians of the 100 out-of-sample years. Sharpe and Sortino use the daily DGS3 rate. Sterling = return / (|MDD| + 10%). '
               'Effective N = 1/Σw² (12 = equal weights on 12 stocks). The diversification ratio is ex-ante (estimation sample).',
          percent=['Ann. return', 'Ann. volatility', 'Max drawdown', 'Ann. return (gross)', 'Max weight'],
          dec1=['Effective N', 'Cost drag (bps)'])

dispersion = load('04_dispersion').rename(columns={'Strategy': PORTFOLIO})
add_table('T16', 'dispersion', 'Dispersion across the 100 experiments: 25th percentile, median and 75th percentile',
          '3. Results', dispersion,
          percent=['Return P25', 'Return median', 'Return P75', 'Return std'])

for number, file_name, benchmark in [('T17', '04_beats_ew', 'EW'), ('T18', '04_beats_group_pick', 'the Group pick'), ('T19', '04_beats_spy', 'SPY')]:
    table = first_column_as_name(load(file_name), PORTFOLIO)
    add_table(number, file_name[3:], f'% of experiments in which each portfolio beats {benchmark} in the same experiment',
              '3. Results', table,
              note='Better = higher Sharpe, return and Sterling; smaller drawdown; lower volatility. 50% = as often better as worse.',
              whole=[column for column in table.columns if column != PORTFOLIO])

rankings = load('04_rankings').rename(columns={'Strategy': PORTFOLIO})
add_table('T20', 'rankings', 'Rank of each portfolio by net Sharpe ratio in each experiment (1 = best of 10)',
          '3. Results', rankings,
          dec1=['Average rank', 'Median rank'], whole=['% ranked 1st', '% in top 3', '% ranked last'])

pairwise_sharpe = first_column_as_name(load('04_pairwise_sharpe'), 'Portfolio A (row)')
add_table('T21', 'pairwise_sharpe', 'Head-to-head: % of experiments in which A (row) has a higher net Sharpe ratio than B (column)',
          '3. Results: head-to-head comparison', pairwise_sharpe,
          note='Each cell compares two portfolios in the same 100 experiments; cell(A,B) + cell(B,A) = 100.',
          whole=[column for column in pairwise_sharpe.columns if column != 'Portfolio A (row)'])

average_win_rate = first_column_as_name(load('04_pairwise_avg_win_rate'), PORTFOLIO)
add_table('T22', 'pairwise_avg_win_rate', 'Average % of head-to-head comparisons won (against the other 9 portfolios)',
          '3. Results: head-to-head comparison', average_win_rate,
          whole=[column for column in average_win_rate.columns if column != PORTFOLIO])

median_difference = first_column_as_name(load('04_pairwise_median_sharpe_diff'), 'Portfolio A (row)')
add_table('T23', 'pairwise_median_sharpe_diff', 'Median paired difference in net Sharpe ratio: A (row) minus B (column)',
          '3. Results: head-to-head comparison', median_difference)

capm = first_column_as_name(load('04_capm_median'), PORTFOLIO)
add_table('T24', 'capm', "Market risk against SPY: CAPM beta, Jensen's alpha, tracking error and information ratio (medians)",
          '3. Results: market risk (CAPM)', capm,
          note="Regression of the daily excess returns of each portfolio on those of SPY in each out-of-sample year (net of costs). "
               "Alpha annualised = daily alpha x 252. OLS t-statistics are only indicative (volatility clustering).",
          percent=['Alpha (ann.)', 'Tracking error'],
          whole=['% beta < 1', '% alpha > 0', '% alpha significant > 0 (t > 1.96)', '% IR > 0 (beats SPY)'])

### 2.4 Robustness checks (notebook 04, sections 10-12)

In [7]:
sharpe_by_cost = first_column_as_name(load('04_sharpe_by_cost'), PORTFOLIO)
add_table('T25', 'sharpe_by_cost', 'Median net Sharpe ratio with different transaction costs',
          'Robustness: transaction costs', sharpe_by_cost, dec3=['0 bps', '10 bps', '25 bps', '50 bps'])

ranking_by_cost = first_column_as_name(load('04_ranking_by_cost'), PORTFOLIO)
add_table('T26', 'ranking_by_cost', 'Rank by median net Sharpe ratio with different transaction costs (1 = best)',
          'Robustness: transaction costs', ranking_by_cost, whole=['0 bps', '10 bps', '25 bps', '50 bps'])

quarterly = load('04_rebalance_semiann_vs_quarterly').rename(columns={'Strategy': PORTFOLIO})
add_table('T27', 'semiannual_vs_quarterly', 'Semi-annual vs quarterly rebalancing (medians, net of 10 bps)',
          'Robustness: rebalancing frequency', quarterly,
          percent=['Return (semi-annual)', 'Return (quarterly)'], whole=['% beats EW Sharpe (quarterly)'])

lambda_table = load('04_lambda_sensitivity').rename(columns={'Strategy': PORTFOLIO})
add_table('T28', 'lambda_sensitivity', 'Markowitz risk aversion: λ = 1, 3 and 5 between Max Sharpe, GMV and EW (medians)',
          'Robustness: Markowitz risk aversion', lambda_table,
          percent=['Ann. return', 'Ann. volatility', 'Max drawdown'], dec1=['Effective N'])

subperiod = load('04_sharpe_by_subperiod').rename(columns={'Strategy': PORTFOLIO})
add_table('T29', 'sharpe_by_subperiod', 'Median net Sharpe ratio by sub-period of the evaluation year',
          'Robustness: dependence on the period', subperiod)

regime = load('04_sharpe_by_regime').rename(columns={'Strategy': PORTFOLIO, 'SPY down': 'Sharpe: SPY down', 'SPY up': 'Sharpe: SPY up'})
regime_return = load('04_return_by_regime')
regime['Return: SPY down'] = regime_return['SPY down']
regime['Return: SPY up'] = regime_return['SPY up']
add_table('T30', 'market_regime', 'Median Sharpe ratio and return in the experiments whose out-of-sample year had SPY down or up',
          'Robustness: dependence on the period', regime,
          note='"SPY down" = 12 of the 100 experiments, with a negative SPY return in the out-of-sample year; they overlap and come from about four '
               'episodes (2015-16, 2018, 2019-20, 2021-23), so they are far fewer than 12 independent observations.',
          percent=['Return: SPY down', 'Return: SPY up'])

# This table has two header rows in the CSV (version, measure): we join them into one name, e.g. "12 stocks (main): Median Sharpe"
subset = pd.read_csv(f'{RESULTS}/04_subset_comparison.csv', encoding='utf-8-sig', header=[0, 1], index_col=0)
subset.columns = [f'{version}: {measure}' for version, measure in subset.columns]
subset = subset.reset_index().rename(columns={subset.index.name if subset.index.name else 'index': PORTFOLIO})
subset = subset.rename(columns={subset.columns[0]: PORTFOLIO})
add_table('T31', 'subset_comparison', 'Do the results depend on the stocks drawn? Main draw, new draw and all eligible stocks',
          'Robustness: dependence on the asset subset', subset,
          note='Same 100 periods. In "All eligible stocks", EW and the Group pick are the same portfolio (so they share the rank and the Group pick never "beats" EW).',
          percent=[column for column in subset.columns if 'Median return' in column],
          dec1=[column for column in subset.columns if 'Rank' in column],
          whole=[column for column in subset.columns if '% beats EW' in column])

universe23 = load('04_universe23_median_metrics').rename(columns={'Strategy': PORTFOLIO})
universe23['Cost drag (bps)'] = universe23['Cost drag'] * 10000
universe23 = universe23.drop(columns=['Cost drag'])
add_table('T32', 'universe23_medians', 'The strategies applied to the group\'s 23 stocks (medians, net of 10 bps)',
          'Robustness: the 23-stock universe', universe23,
          note='Here every strategy holds the same 23 stocks as the Group pick (EW on 23 stocks = the Group pick).',
          percent=['Ann. return', 'Ann. volatility', 'Max drawdown', 'Ann. return (gross)', 'Max weight'], dec1=['Effective N', 'Cost drag (bps)'])

universe23_beats = first_column_as_name(load('04_universe23_beats_group_pick'), PORTFOLIO)
add_table('T33', 'universe23_beats_group_pick', 'On the 23-stock universe: % of experiments in which each strategy beats the Group pick',
          'Robustness: the 23-stock universe', universe23_beats,
          whole=[column for column in universe23_beats.columns if column.startswith('%')])

fullsample = load('04_fullsample_metrics')
add_table('T34', 'fullsample_metrics', 'Full sample 2010-2026: Group pick vs SPY vs ^GSPC (descriptive only - biased by hindsight)',
          'Robustness / limitations: full sample', fullsample,
          note='The 23 stocks were chosen in 2026, so this is NOT out-of-sample evidence. Sterling uses the average of the 17 calendar-year '
               'maximum drawdowns (2026 = year to date). ^GSPC is a price index (no dividends).',
          percent=['CAGR', 'Ann. volatility', 'Max drawdown', 'Avg. calendar-year max drawdown'], dec1=['Final value of 1 USD'])

calendar_years = load('04_calendar_year_returns')
add_table('T35', 'calendar_year_returns', 'Calendar-year returns: Group pick vs SPY vs ^GSPC (2026 = year to date, until 18 September)',
          'Robustness / limitations: full sample', calendar_years,
          percent=[column for column in calendar_years.columns if column != 'Year'], whole=['Year'])

failures = load('04_optimisation_failures')
failures_summary = pd.DataFrame([['Optimisations that failed twice (equal weights used)', len(failures)]],
                                columns=['Item', 'Number'])
add_table('T36', 'optimisation_failures', 'Optimisation failures in all the runs of the backtest',
          'Methods: optimisation failures', failures_summary,
          note='The brief asks to document failures. Each failed SLSQP optimisation is retried once with a looser tolerance; '
               'if it fails again, equal weights are used and the failure is logged in results/04_optimisation_failures.csv.',
          whole=['Number'])

### 2.5 Trade-offs for the conclusion (return, downside risk, concentration, turnover and complexity)

The brief asks to discuss these trade-offs. This table puts them side by side: the numbers come from tables T15 and T17, the last two
columns describe what each portfolio needs to estimate and how complex it is to run.

In [8]:
medians_table = load('04_median_metrics').set_index('Strategy')
beats_ew_table = load('04_beats_ew').set_index(load('04_beats_ew').columns[0])

complexity = {
    'EW': ('nothing', 'Low'),
    'Markowitz (λ=3)': ('expected returns, covariances, λ (optimiser)', 'High'),
    'GMV': ('covariances (optimiser)', 'Medium'),
    'Max Sharpe': ('expected returns, covariances, r_f (optimiser)', 'High'),
    'Inverse vol': ('volatilities only (simple formula)', 'Low'),
    'Risk parity (ERC)': ('covariances (optimiser)', 'Medium'),
    'Most diversified': ('covariances (optimiser)', 'Medium'),
    'Max decorrelation': ('correlations (optimiser)', 'Medium'),
    'Group pick (23 stocks)': ('nothing (but the stock selection was made with hindsight)', 'Low'),
    'SPY (buy & hold)': ('nothing (one ETF)', 'Lowest'),
}

rows = []
for name, (estimated, level) in complexity.items():
    rows.append({
        PORTFOLIO: name,
        'Median return': medians_table.loc[name, 'Ann. return'],
        'Median volatility': medians_table.loc[name, 'Ann. volatility'],
        'Median max drawdown': medians_table.loc[name, 'Max drawdown'],
        'Median Sharpe': medians_table.loc[name, 'Sharpe'],
        '% beats EW (Sharpe)': beats_ew_table.loc[name, '% beats EW: Sharpe'] if name in beats_ew_table.index else float('nan'),
        'Effective N': medians_table.loc[name, 'Effective N'],
        'Turnover (annual)': medians_table.loc[name, 'Turnover (annual)'],
        'What is estimated': estimated,
        'Complexity': level,
    })
tradeoffs = pd.DataFrame(rows)
add_table('T37', 'tradeoffs', 'Trade-offs: return, downside risk, concentration, turnover and complexity (medians, net of 10 bps)',
          '6. Conclusions and recommendation', tradeoffs,
          note='Effective N = 1/Σw² (12 = equal weights on the 12 stocks of each experiment; the Group pick holds 22-23). Complexity = how much must be '
               'estimated and optimised at each rebalancing.',
          percent=['Median return', 'Median volatility', 'Median max drawdown'], whole=['% beats EW (Sharpe)'], dec1=['Effective N'])

### 2.5 Appendix tables

In [9]:
all_assets = load('03_descriptive_stats_all_assets')
add_table('TA01', 'descriptive_all_assets', 'Descriptive statistics of all 25 series (23 stocks, SPY, ^GSPC) at three frequencies',
          'Appendix: data', all_assets,
          percent2=['Mean', 'Std. dev.', 'Min', 'Max'], percent=['Ann. mean', 'Ann. volatility'], whole=['N'], pvalue=['JB p-value'])

design = load('04_experiment_design').drop(columns=['start_index'])
add_table('TA02', 'experiment_design', 'The 100 experiments: periods and the 12 stocks drawn (seed 2026)',
          'Appendix: backtest design', design, whole=['Experiment', 'N eligible'])

weights = first_column_as_name(load('04_weights_illustration'), 'Stock')
add_table('TA03', 'weights_illustration', 'Illustration: weights of each strategy with the last 12 months of data (23 stocks)',
          'Appendix: strategies', weights, percent=[column for column in weights.columns if column != 'Stock'])

characteristics = load('04_illustration_characteristics').rename(columns={'Strategy': PORTFOLIO})
add_table('TA04', 'illustration_characteristics', 'Illustration: ex-ante characteristics of each strategy (last 12 months, 23 stocks)',
          'Appendix: strategies', characteristics,
          percent=['Exp. return', 'Volatility', 'Max weight'], dec1=['Effective N (1/Σw²)'], whole=['N assets with w > 1%'])

iv_erc = first_column_as_name(load('04_iv_vs_erc'), 'Stock')
add_table('TA05', 'iv_vs_erc', 'Inverse volatility vs ERC: weights and risk contributions (last 12 months, 23 stocks)',
          'Appendix: strategies', iv_erc,
          percent=['Weight IV', 'Risk contrib. IV', 'Weight ERC', 'Risk contrib. ERC'])

average_weight = first_column_as_name(load('04_avg_weight_by_stock'), 'Stock')
add_table('TA06', 'avg_weight_by_stock', 'Average weight of each stock when it is drawn, by strategy',
          'Appendix: weights', average_weight, percent=[column for column in average_weight.columns if column != 'Stock'])

for number, file_name, metric in [('TA07', '04_pairwise_return', 'a higher return'),
                                  ('TA08', '04_pairwise_max_drawdown', 'a smaller maximum drawdown'),
                                  ('TA09', '04_pairwise_volatility', 'a lower volatility'),
                                  ('TA10', '04_pairwise_sterling', 'a higher Sterling ratio')]:
    table = first_column_as_name(load(file_name), 'Portfolio A (row)')
    add_table(number, file_name[3:], f'Head-to-head: % of experiments in which A (row) has {metric} than B (column)',
              'Appendix: head-to-head comparison', table,
              whole=[column for column in table.columns if column != 'Portfolio A (row)'])

universe23_pairwise = first_column_as_name(load('04_universe23_pairwise_sharpe'), 'Portfolio A (row)')
add_table('TA11', 'universe23_pairwise_sharpe', 'On the 23-stock universe: head-to-head comparison by net Sharpe ratio',
          'Appendix: the 23-stock universe', universe23_pairwise,
          whole=[column for column in universe23_pairwise.columns if column != 'Portfolio A (row)'])

print('Number of tables:', len(report_tables))

Number of tables: 48


## 3. Saving the tables: Excel (formatted), CSV, Markdown and PNG images

- **Excel:** the numbers stay numbers (so you can still calculate with them); only the way they are **shown** changes
  (number formats). Each sheet has the title in row 1, the note in row 2 and the table from row 3. The first sheet is an index.
- **CSV and Markdown:** the numbers are converted to **text** already rounded (e.g. "35.6%"), ready to paste into Word.
- **PNG images** (`tables/png/`): the same rounded tables drawn as pictures with the title above and the note below, in the colours
  of `appm_style.py` (navy header, light stripes, the Group pick row highlighted). Insert them in Word like a figure.

In [10]:
excel_formats = {'percent': '0.0%', 'percent2': '0.00%', 'whole': '0', 'pvalue': '0.000',
                 'dec1': '0.0', 'dec3': '0.000', 'dec4': '0.0000', 'default': '0.00'}


def format_value(value, kind):
    # Turns one number into the text shown in the CSV / Markdown tables
    text = number_as_text(value, kind)
    if text.startswith('-') and text.strip('-0.%') == '':     # e.g. '-0.0000' -> '0.0000' (a tiny negative number rounded to zero)
        text = text[1:]
    return text


def number_as_text(value, kind):
    if isinstance(value, str):
        return value
    if pd.isna(value):
        return ''
    if isinstance(value, bool):
        return 'Yes' if value else 'No'
    if kind == 'percent':
        return f'{value:.1%}'
    if kind == 'percent2':
        return f'{value:.2%}'
    if kind == 'whole':
        return f'{value:.0f}'
    if kind == 'pvalue':
        return '<0.001' if value < 0.001 else f'{value:.3f}'
    if kind == 'dec1':
        return f'{value:.1f}'
    if kind == 'dec3':
        return f'{value:.3f}'
    if kind == 'dec4':
        return f'{value:.4f}'
    return f'{value:.2f}'


def as_text(entry):
    # A copy of the table in which every value is text with the right format
    table = entry['table']
    text_table = pd.DataFrame(index=table.index)
    for column in table.columns:
        kind = entry['formats'].get(column, 'default')
        text_table[column] = [format_value(value, kind) for value in table[column]]
    return text_table


def markdown_table(text_table):
    # A simple Markdown table (one line per row, columns separated by |)
    lines = ['| ' + ' | '.join(str(column).replace('|', '\\|') for column in text_table.columns) + ' |',
             '|' + '---|' * len(text_table.columns)]
    for _, row in text_table.iterrows():
        lines.append('| ' + ' | '.join(str(value).replace('|', '\\|') for value in row) + ' |')
    return '\n'.join(lines)


# 1) CSV (formatted text) and one Markdown file with every table
markdown_parts = ['# All the tables of the report pack\n',
                  'Created by `notebooks/05_report_pack.ipynb` from the files in `results/`. Rounded for the report; '
                  'the exact values are in `results/` and in `report_tables.xlsx`.\n']
for entry in report_tables:
    text_table = as_text(entry)
    text_table.to_csv(f"{PACK}/tables/{entry['number']}_{entry['name']}.csv", index=False, encoding='utf-8-sig')
    markdown_parts.append(f"## {entry['number']}. {entry['title']}\n")
    markdown_parts.append(f"*Report section: {entry['section']}*\n")
    markdown_parts.append(markdown_table(text_table) + '\n')
    if entry['note']:
        markdown_parts.append(f"Note: {entry['note']}\n")

with open(f'{PACK}/tables/all_tables.md', 'w', encoding='utf-8') as file:
    file.write('\n'.join(markdown_parts))

# 2) PNG image of every table (title above, note below), drawn by save_table_png of appm_style.py.
# The column headers are wrapped after about 11 characters, so that the columns of numbers stay narrow and the image fits a page.
# A few tables with long text get another line length for their text columns (characters per line; the default is 42):
# shorter lines make a wide table narrower, longer lines make a tall table shorter (TA02: the 12 tickers on one line).
text_width = {'T13': 30, 'T14': 60, 'T37': 30, 'TA02': 68}
for entry in report_tables:
    save_table_png(as_text(entry), f"{PACK}/tables/png/{entry['number']}_{entry['name']}.png",
                   title=f"{entry['number']}. {entry['title']}", note=entry['note'],
                   max_header_width=11, max_text_width=text_width.get(entry['number'], 42))

# 3) Excel with formats
index_sheet = pd.DataFrame([[entry['number'], entry['title'], entry['section']] for entry in report_tables],
                           columns=['Table', 'Title', 'Report section'])
sheet_names = {}

with pd.ExcelWriter(f'{PACK}/tables/report_tables.xlsx', engine='openpyxl') as excel:
    index_sheet.to_excel(excel, sheet_name='Index', index=False)
    excel.sheets['Index'].column_dimensions['A'].width = 8
    excel.sheets['Index'].column_dimensions['B'].width = 110
    excel.sheets['Index'].column_dimensions['C'].width = 55

    for entry in report_tables:
        sheet_name = f"{entry['number']} {entry['name']}"[:31]            # Excel: at most 31 characters
        sheet_names[entry['number']] = sheet_name
        table = entry['table']
        text_table = as_text(entry)                                            # only used to choose the column widths
        table.to_excel(excel, sheet_name=sheet_name, index=False, startrow=2)   # the table starts in row 3
        sheet = excel.sheets[sheet_name]

        sheet['A1'] = f"{entry['number']}. {entry['title']}"
        sheet['A1'].font = Font(bold=True, size=12)
        sheet['A2'] = entry['note']
        sheet['A2'].font = Font(italic=True, size=9)

        for column_number, column in enumerate(table.columns, start=1):
            header_cell = sheet.cell(row=3, column=column_number)
            header_cell.font = Font(bold=True)
            header_cell.alignment = Alignment(wrap_text=True, vertical='top')
            kind = entry['formats'].get(column, 'default')
            for row_number in range(4, 4 + len(table)):
                cell = sheet.cell(row=row_number, column=column_number)
                if isinstance(cell.value, (int, float)) and not isinstance(cell.value, bool):
                    cell.number_format = excel_formats[kind]
            # Column width: enough for the longest value as it is shown (rounded); long headers wrap onto two lines
            longest_value = max([len(text) for text in text_table[column]] + [0])
            width = max(longest_value + 2, min(len(str(column)), 16), 8)
            sheet.column_dimensions[header_cell.column_letter].width = min(width, 60)
        sheet.freeze_panes = 'B4'

print('Saved:', f'{PACK}/tables/report_tables.xlsx', '|', len(report_tables), 'tables + index sheet')
print('Saved:', len(report_tables), 'CSV files and all_tables.md in', f'{PACK}/tables/')
print('Saved:', len(os.listdir(f'{PACK}/tables/png')), 'table images in', f'{PACK}/tables/png/')

Saved: ../report_pack/tables/report_tables.xlsx | 48 tables + index sheet
Saved: 48 CSV files and all_tables.md in ../report_pack/tables/
Saved: 48 table images in ../report_pack/tables/png/


## 4. The index of the report pack (`report_pack/INDEX.md`)

In [11]:
lines = ['# Report pack - index of figures and tables', '',
         'Created automatically by `notebooks/05_report_pack.ipynb` (do not edit by hand: run the notebook again).',
         'Figures are in `figures/` (numbered in the order of the report); tables are in `tables/report_tables.xlsx` '
         '(one sheet per table), `tables/<number>_<name>.csv` and `tables/all_tables.md`.',
         'Every table is also an image in `tables/png/` (`<number>_<name>.png`: title, table and note, in the same colours as the '
         'figures), ready to insert in Word. The colours of all the figures and tables are set in `notebooks/appm_style.py`.',
         'The written guide with the key findings and numbers is `REPORT_NOTES.md`.',
         'F = figure and T = table for the main text; FA and TA = appendix. The "report section" labels follow the numbering of the '
         'group\'s draft (1. universe and data, 2.1 stylised facts, 3. strategies, backtest and results); in `REPORT_NOTES.md` the draft\'s '
         'section 3 is split into methods (section 3), results (section 4) and robustness (section 5).', '',
         '## Figures', '',
         '| No. | File | Report section | Caption (suggested) | Made by | Main / Appendix |',
         '|---|---|---|---|---|---|']
for number, file_name, section, caption, made_by, place in figure_list:
    caption = caption.replace('|', '\\|')          # a '|' inside a Markdown table would start a new column
    lines.append(f'| {number} | `figures/{number}_{file_name}` | {section} | {caption} | {made_by} | {place} |')

lines += ['', '## Tables', '',
          '| No. | Title | Report section | Excel sheet | CSV | PNG |',
          '|---|---|---|---|---|---|']
for entry in report_tables:
    title = entry['title'].replace('|', '\\|')
    lines.append(f"| {entry['number']} | {title} | {entry['section']} | `{sheet_names[entry['number']]}` | "
                 f"`tables/{entry['number']}_{entry['name']}.csv` | `tables/png/{entry['number']}_{entry['name']}.png` |")

with open(f'{PACK}/INDEX.md', 'w', encoding='utf-8') as file:
    file.write('\n'.join(lines) + '\n')

print('Saved:', f'{PACK}/INDEX.md')
print(f'{len(figure_list)} figures and {len(report_tables)} tables in the report pack')

Saved: ../report_pack/INDEX.md
40 figures and 48 tables in the report pack


In [12]:
# A quick look at the most important table of the report (T15), as it will appear in the report
display(as_text(report_tables[[entry['number'] for entry in report_tables].index('T15')]))

,Portfolio,Ann. return,Ann. volatility,Sharpe,Sortino,Max drawdown,Calmar,Sterling,Ann. return (gross),Sharpe (gross),Turnover (annual),Effective N,Max weight,Diversification ratio (ex-ante),Cost drag (bps)
0,EW,31.2%,16.9%,1.81,2.73,-11.1%,3.30,1.68,31.2%,1.81,0.27,12.0,8.3%,1.63,3.5
1,Markowitz (λ=3),39.8%,34.5%,1.23,1.87,-23.5%,2.09,1.37,40.1%,1.24,2.21,1.5,76.9%,1.11,32.6
2,GMV,17.3%,12.8%,1.34,1.90,-9.8%,2.01,0.93,17.4%,1.35,0.95,4.4,35.0%,1.55,10.9
3,Max Sharpe,27.9%,20.8%,1.29,2.02,-14.4%,2.19,1.28,28.3%,1.31,2.14,2.8,49.6%,1.40,27.0
4,Inverse vol,25.7%,14.7%,1.78,2.64,-9.8%,3.16,1.47,25.7%,1.78,0.27,10.9,12.8%,1.64,3.4
5,Risk parity (ERC),26.5%,14.4%,1.80,2.71,-10.0%,3.22,1.48,26.5%,1.80,0.31,10.7,13.8%,1.69,3.8
6,Most diversified,28.7%,15.2%,1.70,2.51,-10.7%,3.05,1.41,28.8%,1.71,0.84,6.3,26.5%,1.79,10.1
7,Max decorrelation,33.3%,19.0%,1.66,2.48,-13.1%,2.83,1.58,33.4%,1.66,0.86,7.1,21.0%,1.72,10.8
8,Group pick (23 stocks),35.6%,15.8%,1.70,2.74,-10.1%,2.98,1.60,35.6%,1.71,0.28,23.0,4.3%,1.70,3.8
9,SPY (buy & hold),17.7%,14.0%,1.13,1.66,-9.4%,2.00,0.89,17.7%,1.13,0.00,,,,0.0
